# CS224N · Lecture 1 — Introduction and Word Vectors

**Course:** Stanford CS224N, *Natural Language Processing with Deep Learning* (Christopher Manning)  
**Slides:** [cs224n-spr2024-lecture01-wordvecs1.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture01-wordvecs1.pdf)

---

### The one big idea of this lecture

> **Word meaning can be represented surprisingly well by a dense vector of real numbers.**

Everything else in modern NLP, including Transformers and LLMs, is built on top of this idea. An LLM's very first layer is literally a table of word (token) vectors, the same object we build here.

### What you will be able to do after this notebook

1. Explain why one-hot vectors and dictionaries like WordNet are poor representations of meaning.
2. State the *distributional hypothesis* and turn a corpus into (center, context) training pairs.
3. Write down the **skip-gram word2vec** objective and its softmax probability model.
4. **Derive the gradients by hand** (this was done on the whiteboard in lecture, and the slides are blank there; it is fully written out here).
5. Implement gradient descent, SGD and mini-batch SGD, and train word2vec from scratch in NumPy.
6. Explore real pretrained vectors (similarity, analogies, visualization).

### How to use this notebook
* Run it top to bottom (Google Colab or local Jupyter). Cells marked **`# [PyTorch]`** need PyTorch; cells marked **`# [Optional]`** download data or need extra packages and can be skipped.
* Sections marked **🔎 Beyond the slides** add material that is not in the lecture but that an ML engineer should know.
* Each section ends with a short **✅ Check yourself** question. Answers are hidden in collapsible blocks.

### Map of this series (Lectures 1–4)
| Notebook | Topic | Core skill |
|---|---|---|
| **L1 (this one)** | Word vectors, word2vec | Representations + the softmax objective |
| L2 | Word vectors 2, GloVe, evaluation, senses, neural classifiers | Negative sampling, count-based methods, classification |
| L3 | Matrix calculus and backpropagation | Computing gradients by hand and algorithmically |
| L4 | Dependency parsing | Linguistic structure + a neural parser |

In [ ]:
# Setup: only NumPy + Matplotlib are needed for the core of this notebook.
import numpy as np
import matplotlib.pyplot as plt
import random, math, collections

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)
print("numpy", np.__version__)

---
## 1. Why NLP, and why is it hard?

The lecture opens with a tour of what neural NLP can do: neural machine translation (e.g., English → Swahili), free-text question answering ("when did Kendrick Lamar's first album come out?" → "July 2, 2011"), GPT-2 writing a fluent (made-up!) news article from a two-sentence prompt, ChatGPT/GPT-4, and text-to-image models like DALL-E 2.

Why is human language hard for a computer?

* **It is ambiguous.** "Scientists count whales from space": who is in space? (We study this in Lecture 4.)
* **Meaning depends on context.** "bank" in *river bank* vs. *bank loan*.
* **It is a social system that keeps changing.** New words (*rizz*) and new meanings (*wicked* = good) appear all the time.
* **It is extremely sparse.** Most word combinations you will ever see are ones you have never seen before.

The deep-learning answer to all of this starts with the same move: **learn representations from data instead of writing them by hand.**

#### A 60-second history (useful context for the whole course)
| Era | Idea | Where in CS224N |
|---|---|---|
| 1950s–1980s | Hand-written rules and grammars | L4 (grammar), L18 |
| 1990s–2000s | Statistical NLP: counts, n-grams, feature engineering + linear classifiers | L2, L4, L5 |
| 2013 | **word2vec**: dense word vectors learned by prediction | **L1**, L2 |
| 2014–2016 | RNNs, LSTMs, seq2seq, attention | L5, L6 |
| 2017 | Transformers | L8 |
| 2018+ | Pretraining (BERT, GPT), then LLMs, instruction tuning, RLHF | L9, L10 |

---
## 2. How do we represent the meaning of a word?

### 2.1 Denotational semantics
The classical linguistic view: a word is a **signifier** (symbol) that points to a **signified** (an idea or a set of things in the world).

$$\text{tree} \iff \{🌳, 🌲, 🌴, \dots\}$$

This is *denotational semantics*. It is a fine philosophy, but it doesn't directly give a computer anything it can compute with.

### 2.2 The old NLP solution: WordNet
WordNet is a hand-built thesaurus. It stores **synonym sets** ("synsets") and **hypernyms** ("is-a" relations: panda → procyonid → carnivore → … → entity). The slide shows this code using NLTK:

In [ ]:
# [Optional] needs nltk + the WordNet corpus (downloads ~10MB). Safe to skip.
try:
    import nltk
    nltk.download("wordnet", quiet=True)
    from nltk.corpus import wordnet as wn

    poses = {'n': 'noun', 'v': 'verb', 's': 'adj (s)', 'a': 'adj', 'r': 'adv'}
    print("Synsets containing 'good':")
    for synset in wn.synsets("good")[:10]:
        print("  {}: {}".format(poses[synset.pos()], ", ".join(l.name() for l in synset.lemmas())))

    panda = wn.synset("panda.n.01")
    print("\nHypernyms of 'panda':")
    print(" ->", "\n -> ".join(s.name() for s in panda.closure(lambda s: s.hypernyms())))
except Exception as e:
    print("Skipping WordNet demo:", e)

### 2.3 Problems with resources like WordNet
* **Missing nuance.** "proficient" is listed as a synonym of "good", which is true only in some contexts.
* **Missing new meanings** (*wicked, badass, nifty, ninja*) and **impossible to keep up to date**.
* **Subjective and costly**: it needs human labor to create and adapt (and to other languages).
* It even lists offensive synonyms without any notion of connotation.
* **Crucially: it can't give you a graded similarity score** between any two words, which is what downstream systems need.

### 2.4 Words as discrete symbols: one-hot vectors
In traditional NLP, a word is an atomic symbol, a *localist* representation. Encoded as a vector, word number $i$ in a vocabulary of size $|V|$ is the one-hot vector $e_i \in \{0,1\}^{|V|}$:

```
motel = [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
hotel = [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
```

**The problem:** If a user searches for "Seattle motel", we want to match documents with "Seattle hotel". But any two different one-hot vectors are **orthogonal**, so their similarity is exactly zero. There is *no* natural notion of similarity. And the dimension is huge (500,000+ for a realistic English vocabulary).

In [ ]:
vocab_demo = ["the", "a", "seattle", "hotel", "motel", "cat", "dog"]
idx = {w: i for i, w in enumerate(vocab_demo)}

def one_hot(word):
    v = np.zeros(len(vocab_demo))
    v[idx[word]] = 1.0
    return v

def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

print("motel:", one_hot("motel"))
print("hotel:", one_hot("hotel"))
print("cosine(hotel, motel) =", cosine(one_hot("hotel"), one_hot("motel")))
print("cosine(hotel, cat)   =", cosine(one_hot("hotel"), one_hot("cat")))
print("-> one-hot vectors say 'hotel' is as different from 'motel' as it is from 'cat'.")

**Solution proposed in lecture:** don't rely on a hand-built synonym list (it fails due to incompleteness). Instead, **learn to encode similarity in the vectors themselves.**

### 2.5 Distributional semantics: "You shall know a word by the company it keeps" (J. R. Firth, 1957)

> A word's meaning is given by the words that frequently appear close by.

This is one of the most successful ideas in all of statistical NLP. When word $w$ appears in text, its **context** is the set of words within a fixed-size window. We use the *many* contexts of $w$ to build up its representation:

```
…government debt problems turning into  banking  crises as happened in 2009…
…saying that Europe needs unified       banking  regulation to replace the hodgepodge…
…India has just given its               banking  system a shot in the arm…
```
Those surrounding words will represent **banking**. Let's extract contexts in code:

In [ ]:
text = ("government debt problems turning into banking crises as happened in 2009 . "
        "saying that europe needs unified banking regulation to replace the hodgepodge . "
        "india has just given its banking system a shot in the arm .").split()

def contexts(tokens, target, m=3):
    # Return the window of m words on each side of every occurrence of `target`.
    out = []
    for t, w in enumerate(tokens):
        if w == target:
            left = tokens[max(0, t - m):t]
            right = tokens[t + 1:t + 1 + m]
            out.append((left, right))
    return out

for left, right in contexts(text, "banking"):
    print(f"{' '.join(left):>30} [banking] {' '.join(right)}")

### 2.6 Word vectors (a.k.a. word embeddings)
We will build a **dense** vector for each word, chosen so that it is **similar to vectors of words that appear in similar contexts**, with similarity measured by the **dot product**.

Example from the slides (8 dimensions shown; real ones have 50–1000):

In [ ]:
banking  = np.array([0.286, 0.792, -0.177, -0.107, 0.109, -0.542, 0.349, 0.271])
monetary = np.array([0.413, 0.582, -0.007,  0.247, 0.216, -0.718, 0.147, 0.051])
random_v = rng.normal(size=8) * 0.4

print("dot(banking, monetary)    =", round(float(banking @ monetary), 3))
print("cosine(banking, monetary) =", round(float(cosine(banking, monetary)), 3))
print("cosine(banking, random)   =", round(float(cosine(banking, random_v)), 3))

Terminology: word vectors = **word embeddings** = **(neural) word representations**. They are a **distributed** representation: the meaning is spread across all dimensions, and each dimension takes part in representing many words. This contrasts with the *localist* one-hot representation, where each dimension means exactly one word.

> **Dot product vs. cosine.** Word2vec's math uses the raw dot product $u^\top v$. When we *compare* trained vectors we usually use **cosine similarity** $\frac{u^\top v}{\|u\|\|v\|}$. That removes the effect of vector length, which tends to correlate with word frequency.

<details><summary>✅ <b>Check yourself:</b> Why is a 300-dim dense vector better than a 500,000-dim one-hot vector, beyond saving memory?</summary>

Because similar words get similar vectors, a model that learns something about "hotel" automatically generalizes some of it to "motel". With one-hot vectors every word is an island: a classifier trained on "hotel" learns nothing about "motel". Dense vectors give **generalization through shared structure**.
</details>

---
## 3. Word2vec: overview (Mikolov et al., 2013)

Word2vec is a **framework for learning word vectors**. The idea:

1. We have a large corpus ("body") of text: a long list of words.
2. Every word in a fixed vocabulary is represented by a vector (initialized randomly).
3. Go through each position $t$ in the text, which has a **center word** $c$ and **context ("outside") words** $o$.
4. Use the **similarity of the vectors** for $c$ and $o$ to compute the **probability of $o$ given $c$** (or vice versa).
5. **Keep adjusting the word vectors to maximize this probability.**

The variant we study is **skip-gram (SG)**: predict each context word from the center word. With window size $m=2$:

```
… problems   turning   into   banking   crises   as …
             P(w_{t-2}|w_t)  P(w_{t-1}|w_t)  [center w_t]  P(w_{t+1}|w_t)  P(w_{t+2}|w_t)
```

Then we slide the window one position to the right and repeat.

Turning a corpus into skip-gram **training pairs** is simple:

In [ ]:
def skipgram_pairs(tokens, m=2):
    # Yield (center, context) pairs for every position and every offset -m..m (excluding 0).
    pairs = []
    for t, center in enumerate(tokens):
        for j in range(-m, m + 1):
            if j == 0 or not (0 <= t + j < len(tokens)):
                continue
            pairs.append((center, tokens[t + j]))
    return pairs

sentence = "problems turning into banking crises as".split()
for c, o in skipgram_pairs(sentence, m=2):
    if c == "banking":
        print(f"center={c:8s} context={o}")

---
## 4. The word2vec objective function

### 4.1 Likelihood
For each position $t = 1,\dots,T$, predict the context words within a window of fixed size $m$, given the center word $w_t$. The **likelihood** of the data is

$$L(\theta) = \prod_{t=1}^{T} \prod_{\substack{-m \le j \le m \\ j \ne 0}} P(w_{t+j} \mid w_t; \theta)$$

where $\theta$ is *all* the variables to be optimized (all the word vectors).

### 4.2 Objective (loss / cost) = average negative log likelihood

$$J(\theta) = -\frac{1}{T}\log L(\theta) = -\frac{1}{T}\sum_{t=1}^{T}\sum_{\substack{-m \le j \le m \\ j \ne 0}} \log P(w_{t+j}\mid w_t;\theta)$$

**Minimizing $J$ ⟺ maximizing predictive accuracy.** Why these transformations?
* **log** turns a product of many tiny probabilities (which underflows to 0 in floating point) into a sum, and sums are easy to differentiate.
* **negative** because optimizers conventionally *minimize*.
* **$1/T$** makes the loss scale-independent of corpus size (so learning rates transfer).

### 4.3 How do we compute $P(o \mid c)$? Two vectors per word
Each word $w$ gets **two** vectors:
* $v_w$ when $w$ is a **center** word
* $u_w$ when $w$ is a **context (outside)** word

Then for center word $c$ and context word $o$:

$$\boxed{P(o \mid c) = \frac{\exp(u_o^\top v_c)}{\sum_{w \in V}\exp(u_w^\top v_c)}}$$

Read it in three steps:
1. **Dot product** $u_o^\top v_c = \sum_i u_{o,i}v_{c,i}$ compares the similarity of $o$ and $c$. Larger dot product → larger probability.
2. **Exponentiation** makes everything positive.
3. **Normalize** over the entire vocabulary to get a probability distribution.

### 4.4 The softmax function
That formula is an instance of the **softmax** $\mathbb{R}^n \to (0,1)^n$:

$$\text{softmax}(x)_i = \frac{\exp(x_i)}{\sum_{j=1}^{n}\exp(x_j)} = p_i$$

* **"max"** because it amplifies the probability of the largest $x_i$.
* **"soft"** because it still assigns some probability to smaller $x_i$.
* (It is a slightly odd name: it returns a *distribution*, and is really a smooth version of *arg*max.)

#### 🔎 Beyond the slides: implementing softmax safely
A naive implementation overflows: `exp(1000)` is `inf`. Because softmax is invariant to adding a constant to all inputs, $\text{softmax}(x) = \text{softmax}(x - \max_j x_j)$, so we subtract the max first. The same trick gives a stable **log-softmax** via the *log-sum-exp* identity:
$\log \text{softmax}(x)_i = x_i - \max x - \log\sum_j e^{x_j-\max x}$. Every DL framework does this internally (`torch.log_softmax`, and `CrossEntropyLoss` takes raw *logits* for this reason).

In [ ]:
def softmax_naive(x):
    e = np.exp(x)
    return e / e.sum()

def softmax(x, axis=-1):
    # Numerically stable softmax (works on vectors or batches of rows).
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=axis, keepdims=True))

x = np.array([2.0, 1.0, 0.1])
print("softmax([2, 1, 0.1])  =", softmax(x), " sum =", softmax(x).sum())
print("'max' amplifies:  softmax(3*x) =", softmax(3 * x))
print("'soft' keeps mass: softmax(x/3) =", softmax(x / 3))

big = np.array([1000.0, 999.0, 0.0])
with np.errstate(over="ignore", invalid="ignore"):
    print("\nnaive softmax on big inputs ->", softmax_naive(big))
print("stable softmax on big inputs ->", softmax(big))
print("stable log_softmax           ->", log_softmax(big))

### 4.5 How many parameters?
$\theta$ is all the model parameters in one long vector. With $d$-dimensional vectors and $|V|$ words, and two vectors per word:

$$\theta = \begin{bmatrix} v_{\text{aardvark}} \\ \vdots \\ v_{\text{zebra}} \\ u_{\text{aardvark}} \\ \vdots \\ u_{\text{zebra}} \end{bmatrix} \in \mathbb{R}^{2d|V|}$$

In practice we store these as two matrices: $V_{\text{center}} \in \mathbb{R}^{|V|\times d}$ and $U \in \mathbb{R}^{|V|\times d}$ (one **row** per word, which is how DL libraries lay it out).

In [ ]:
for V_size, d in [(10_000, 100), (400_000, 300), (3_000_000, 300)]:
    n = 2 * d * V_size
    print(f"|V|={V_size:>9,}  d={d}:  {n:>13,} parameters  (~{n*4/1e9:.2f} GB as float32)")

---
## 5. Deriving the word2vec gradients (the whiteboard part of the lecture)

In lecture this was done live on the board; the slides are blank here. It is the single most important derivation in Lecture 1, and it reappears in Assignment 2. Let's do it carefully.

We want the gradient of the loss for **one** (center $c$, outside $o$) pair:

$$J_{c,o} = -\log P(o\mid c) = -\log \frac{\exp(u_o^\top v_c)}{\sum_{w\in V}\exp(u_w^\top v_c)} = -u_o^\top v_c + \log \sum_{w \in V}\exp(u_w^\top v_c)$$

### 5.1 Gradient with respect to the center vector $v_c$

**First term:** $\dfrac{\partial}{\partial v_c}\left(-u_o^\top v_c\right) = -u_o$.

**Second term:** use the chain rule. Let $Z = \sum_{w}\exp(u_w^\top v_c)$:

$$\frac{\partial}{\partial v_c}\log Z = \frac{1}{Z}\frac{\partial Z}{\partial v_c} = \frac{1}{Z}\sum_{x\in V}\exp(u_x^\top v_c)\,u_x = \sum_{x \in V}\underbrace{\frac{\exp(u_x^\top v_c)}{\sum_w \exp(u_w^\top v_c)}}_{P(x\mid c)}\,u_x$$

Putting both together:

$$\boxed{\frac{\partial J_{c,o}}{\partial v_c} = -u_o + \sum_{x\in V}P(x\mid c)\,u_x = -\big(\underbrace{u_o}_{\text{observed}} - \underbrace{\mathbb{E}_{x\sim P(\cdot\mid c)}[u_x]}_{\text{expected}}\big)}$$

**Intuition: "observed minus expected".** The gradient step $v_c \leftarrow v_c - \alpha\nabla$ moves $v_c$ **toward** the vector of the word that actually appeared ($u_o$) and **away** from the average of what the model currently predicts. When the model already predicts $o$ with probability 1, the expectation equals $u_o$ and the gradient is zero: nothing to learn.

### 5.2 Gradient with respect to the outside vectors $u_w$
For any word $w$ in the vocabulary:

$$\frac{\partial J_{c,o}}{\partial u_w} = -\mathbb{1}[w=o]\,v_c + P(w\mid c)\,v_c = \big(P(w\mid c) - \mathbb{1}[w=o]\big)\,v_c$$

* For the true context word ($w=o$): $(P(o\mid c) - 1)\,v_c$. Since this coefficient is negative, the update pulls $u_o$ toward $v_c$.
* For every other word: $P(w\mid c)\,v_c$. The update pushes $u_w$ away from $v_c$, in proportion to how much probability it wrongly got.

### 5.3 The compact matrix form (memorize this!)
Stack the outside vectors as rows of $U \in \mathbb{R}^{|V|\times d}$. Let $\hat{y} = \text{softmax}(U v_c) \in \mathbb{R}^{|V|}$ be the predicted distribution and $y$ the one-hot vector of the true word $o$. Then

$$J = -\log \hat{y}_o, \qquad \frac{\partial J}{\partial v_c} = U^\top(\hat{y} - y), \qquad \frac{\partial J}{\partial U} = (\hat{y} - y)\,v_c^\top$$

The term $\hat{y} - y$ (**prediction minus target**) is the gradient of *softmax + cross-entropy* with respect to the logits. You will see it in virtually every classifier in this course (we re-derive it in Lecture 3).

Let's implement this and **verify it numerically**. Never trust a hand-derived gradient until you've checked it!

In [ ]:
def naive_softmax_loss_and_gradient(v_c, o, U):
    # Skip-gram naive-softmax loss for one (center, outside) pair.
    #   v_c : (d,)    center word vector
    #   o   : int     index of the true outside word
    #   U   : (V, d)  outside vectors, one row per word
    # Returns: loss, grad wrt v_c (d,), grad wrt U (V, d)
    y_hat = softmax(U @ v_c)            # (V,) predicted distribution
    loss = -np.log(y_hat[o])
    delta = y_hat.copy()
    delta[o] -= 1.0                     # y_hat - y
    grad_v_c = U.T @ delta              # (d,)
    grad_U = np.outer(delta, v_c)       # (V, d)
    return loss, grad_v_c, grad_U


def numerical_gradient(f, x, h=1e-5):
    # Centered finite differences: (f(x+h) - f(x-h)) / 2h for every entry of x.
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + h; fp = f()
        x[i] = old - h; fm = f()
        x[i] = old
        grad[i] = (fp - fm) / (2 * h)
    return grad


V_size, d = 7, 4
U_test = rng.normal(size=(V_size, d))
v_test = rng.normal(size=d)
o_test = 3

loss, g_v, g_U = naive_softmax_loss_and_gradient(v_test, o_test, U_test)
num_g_v = numerical_gradient(lambda: naive_softmax_loss_and_gradient(v_test, o_test, U_test)[0], v_test)
num_g_U = numerical_gradient(lambda: naive_softmax_loss_and_gradient(v_test, o_test, U_test)[0], U_test)

def rel_err(a, b):
    return np.max(np.abs(a - b) / np.maximum(1e-8, np.abs(a) + np.abs(b)))

print("loss =", round(float(loss), 4))
print("relative error dJ/dv_c:", rel_err(g_v, num_g_v))
print("relative error dJ/dU  :", rel_err(g_U, num_g_U))
print("(anything below ~1e-6 means the analytic gradient is right)")

# Sanity-check the "observed minus expected" formula explicitly:
p = softmax(U_test @ v_test)
print("matches -(u_o - E[u_x]):", np.allclose(g_v, -(U_test[o_test] - p @ U_test)))

<details><summary>✅ <b>Check yourself:</b> What happens to the gradient w.r.t. $v_c$ when the model predicts the true word with probability 1?</summary>

$\hat{y} = y$, so $\hat{y}-y = 0$ and both gradients vanish. Equivalently, $\mathbb{E}[u_x] = u_o$ so "observed − expected" is zero. The loss $-\log 1 = 0$ is at its minimum for that pair.
</details>

<details><summary>✅ <b>Check yourself:</b> Why is this "naive" softmax expensive?</summary>

Computing $\hat{y}$ requires a dot product with **every** word in the vocabulary ($O(|V|d)$ per pair), and the gradient touches **every row** of $U$. With $|V|$ = 1M and billions of pairs that's far too slow. Lecture 2 fixes this with **negative sampling**.
</details>

---
## 6. Optimization basics: gradient descent

We have a cost function $J(\theta)$ we want to minimize. **Gradient descent:** for the current $\theta$, compute the gradient $\nabla_\theta J(\theta)$, then take a small step in the direction of the **negative** gradient. Repeat.

$$\theta^{\text{new}} = \theta^{\text{old}} - \alpha\,\nabla_\theta J(\theta) \qquad\text{(matrix form)}$$
$$\theta_j^{\text{new}} = \theta_j^{\text{old}} - \alpha\,\frac{\partial}{\partial\theta_j^{\text{old}}}J(\theta) \qquad\text{(single parameter)}$$

$\alpha$ is the **step size** or **learning rate**. Our objectives are generally **not convex**, so there's no guarantee of finding the global minimum. "But life turns out to be okay": in practice, for neural nets, gradient methods find good solutions.

```python
while True:
    theta_grad = evaluate_gradient(J, corpus, theta)
    theta = theta - alpha * theta_grad
```

Let's see the effect of the learning rate on a simple 2-D bowl $J(\theta) = \tfrac{1}{2}(\theta_1^2 + 10\,\theta_2^2)$:

In [ ]:
def J_bowl(th):     return 0.5 * (th[0]**2 + 10 * th[1]**2)
def grad_bowl(th):  return np.array([th[0], 10 * th[1]])

def gradient_descent(alpha, steps=30, start=(-9.0, 2.5)):
    th = np.array(start)
    path = [th.copy()]
    for _ in range(steps):
        th = th - alpha * grad_bowl(th)
        path.append(th.copy())
    return np.array(path)

xs, ys = np.meshgrid(np.linspace(-10, 10, 200), np.linspace(-4, 4, 200))
Z = 0.5 * (xs**2 + 10 * ys**2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, alpha, title in zip(axes, [0.02, 0.15, 0.205], ["too small", "good", "too big (diverging)"]):
    path = gradient_descent(alpha)
    ax.contour(xs, ys, Z, levels=20, cmap="Greys", linewidths=0.6)
    ax.plot(path[:, 0], path[:, 1], "o-", ms=3, color="crimson")
    ax.set_title(f"alpha = {alpha}: {title}\nfinal J = {J_bowl(path[-1]):.3g}")
    ax.set_xlim(-10, 10); ax.set_ylim(-4, 4)
plt.tight_layout(); plt.show()

Notice the middle plot: the steep direction ($\theta_2$) oscillates while the shallow one ($\theta_1$) crawls. That's the motivation for **momentum** and **adaptive** methods like Adam (🔎 covered in the L2 notebook).

### 6.1 Stochastic gradient descent (SGD)
**Problem:** $J(\theta)$ is a function of **all** windows in the corpus (potentially billions!). Computing $\nabla_\theta J(\theta)$ is very expensive, and you'd wait a very long time before making a single update. That's a very bad idea for pretty much all neural nets.

**Solution: SGD.** Repeatedly sample windows, and update after each one (or after each small **mini-batch** of, say, 32–512 examples):

```python
while True:
    window = sample_window(corpus)
    theta_grad = evaluate_gradient(J, window, theta)
    theta = theta - alpha * theta_grad
```

The mini-batch gradient is an **unbiased but noisy estimate** of the full gradient. The noise isn't only a cost: it helps escape poor regions and acts as a mild regularizer. Mini-batches also make good use of vectorized hardware (GPUs).

Let's compare full-batch GD and mini-batch SGD on a linear regression problem with 20,000 examples, measuring progress per *example processed* (which is what compute costs):

In [ ]:
N, D = 20_000, 10
X = rng.normal(size=(N, D))
w_true = rng.normal(size=D)
yreg = X @ w_true + 0.1 * rng.normal(size=N)

def mse(w):          return 0.5 * np.mean((X @ w - yreg) ** 2)
def grad(w, idx):    return X[idx].T @ (X[idx] @ w - yreg[idx]) / len(idx)

budget = 5 * N                       # 5 passes worth of examples
results = {}
for name, B, alpha in [("full-batch GD", N, 0.5), ("mini-batch SGD (B=64)", 64, 0.05), ("SGD (B=1)", 1, 0.01)]:
    w = np.zeros(D); seen = 0; curve = []
    while seen < budget:
        idx = np.arange(N) if B == N else rng.integers(0, N, size=B)
        w -= alpha * grad(w, idx)
        seen += B
        curve.append((seen, mse(w)))
    results[name] = np.array(curve)

plt.figure(figsize=(7, 4))
for name, c in results.items():
    plt.plot(c[:, 0] / N, c[:, 1], label=name)
plt.yscale("log"); plt.xlabel("epochs (examples processed / N)"); plt.ylabel("training loss")
plt.legend(); plt.title("Same compute budget: SGD makes many more (noisy) updates"); plt.show()

---
## 7. Training word2vec from scratch (NumPy)

Now we put everything together: corpus → skip-gram pairs → naive-softmax loss → gradients → mini-batch SGD.

Real word2vec is trained on billions of words. To make this run in seconds and still show *real* emergent structure, we use a small **synthetic corpus** generated from templates. It contains royals vs. commoners, male vs. female (via *he/his* vs. *she/her*), animals, and fruits. **The model is never told any of these categories**; it only sees which words occur near which.

In [ ]:
male_royal, female_royal = ["king", "prince"], ["queen", "princess"]
male_common, female_common = ["man", "boy"], ["woman", "girl"]
animals = ["cat", "dog", "horse", "cow"]
fruits = ["apple", "banana", "orange", "mango"]

def person_sentence(w, royal, male):
    he, his = ("he", "his") if male else ("she", "her")
    t = [f"the {w} said {he} was tired", f"the {w} smiled at {his} friend",
         f"{he} is the {w}", f"the {w} lost {his} hat"]
    if royal:
        t += [f"the {w} sat on the throne", f"the {w} wore the crown",
              f"the {w} rules the kingdom", f"the {w} lives in the palace"]
    else:
        t += [f"the {w} works in the village", f"the {w} walked to the market",
              f"the {w} lives in a small house", f"the {w} sold bread at the market"]
    return random.choice(t)

def random_sentence():
    r = random.random()
    if r < 0.6:
        royal, male = random.random() < 0.5, random.random() < 0.5
        pool = (male_royal if male else female_royal) if royal else (male_common if male else female_common)
        return person_sentence(random.choice(pool), royal, male)
    if r < 0.8:
        a = random.choice(animals)
        return random.choice([f"the {a} ate the grass", f"the {a} ran across the field",
                              f"the {a} slept in the barn", f"a {a} is an animal"])
    f = random.choice(fruits)
    return random.choice([f"i ate a sweet {f}", f"the {f} is ripe",
                          f"she bought a {f} at the market", f"a {f} is a fruit"])

random.seed(0)
corpus = [random_sentence().split() for _ in range(6000)]
vocab = sorted({w for s in corpus for w in s})
w2i = {w: i for i, w in enumerate(vocab)}
V_size = len(vocab)

pairs = np.array([(w2i[c], w2i[o]) for s in corpus for c, o in skipgram_pairs(s, m=2)])
print("example sentences:", *[" ".join(s) for s in corpus[:4]], sep="\n  ")
print(f"\nvocabulary size = {V_size}, training pairs = {len(pairs):,}")

**The training loop.** For a mini-batch of $B$ pairs we compute all $B$ predicted distributions at once: scores $S = V_c U^\top \in \mathbb{R}^{B\times|V|}$, then $\hat{Y} = \text{softmax}(S)$, then $\Delta = (\hat{Y} - Y)/B$. The batched gradients follow directly from §5.3:
$\nabla_{V_c} = \Delta U$ and $\nabla_U = \Delta^\top V_c$.

Note `np.add.at`: the same center word can appear several times in a batch, so we must **accumulate** its gradient rows (plain fancy-index assignment would silently drop duplicates; a classic bug).

In [ ]:
def train_word2vec(pairs, V_size, d=20, lr=0.5, batch_size=128, epochs=5, seed=0):
    r = np.random.default_rng(seed)
    # Small random init (as in the original word2vec code).
    Vc = (r.random((V_size, d)) - 0.5) / d      # center vectors v_w (rows)
    U  = (r.random((V_size, d)) - 0.5) / d      # outside vectors u_w (rows)
    history = []
    for ep in range(epochs):
        perm = r.permutation(len(pairs))
        total = 0.0
        for k in range(0, len(pairs), batch_size):
            batch = pairs[perm[k:k + batch_size]]
            c, o = batch[:, 0], batch[:, 1]
            B = len(c)
            vc = Vc[c]                                   # (B, d)
            Y_hat = softmax(vc @ U.T, axis=1)            # (B, V)
            total += -np.log(Y_hat[np.arange(B), o]).sum()
            delta = Y_hat
            delta[np.arange(B), o] -= 1.0                # Y_hat - Y
            delta /= B                                   # average over the batch
            grad_vc = delta @ U                          # (B, d)
            grad_U = delta.T @ vc                        # (V, d)
            U -= lr * grad_U
            np.add.at(Vc, c, -lr * grad_vc)              # accumulate repeated rows!
        history.append(total / len(pairs))
        print(f"epoch {ep+1}: average loss J = {history[-1]:.4f}")
    return Vc, U, history

Vc, U, history = train_word2vec(pairs, V_size)
print(f"\n(uniform guessing would give log|V| = {np.log(V_size):.3f})")

Why can't the loss go to zero? Because skip-gram is a **"bag of words" model**: it makes the *same* prediction for every context position around the center word. Given "king", it must spread probability over *the, sat, wore, said, he, …*. The loss bottoms out at the true entropy of contexts.

**Which vectors do we keep?** We have $v_w$ and $u_w$ for each word. The standard choice (Lecture 2) is to **average** them (or just use $v_w$).

In [ ]:
E = (Vc + U) / 2
E_norm = E / np.linalg.norm(E, axis=1, keepdims=True)

def most_similar(word, k=5):
    sims = E_norm @ E_norm[w2i[word]]
    order = np.argsort(-sims)
    return [(vocab[i], round(float(sims[i]), 2)) for i in order if vocab[i] != word][:k]

def analogy(a, b, c, k=3):
    # a : b :: c : ?   ->  argmax_x cos(x, b - a + c), excluding a, b, c
    target = E_norm[w2i[b]] - E_norm[w2i[a]] + E_norm[w2i[c]]
    sims = E_norm @ target
    for w in (a, b, c):
        sims[w2i[w]] = -np.inf
    return [vocab[i] for i in np.argsort(-sims)[:k]]

for w in ["king", "cat", "apple", "woman", "palace"]:
    print(f"{w:7s} ->", most_similar(w))

print()
for a, b, c in [("man", "king", "woman"), ("boy", "prince", "girl"), ("man", "boy", "woman"), ("king", "queen", "prince")]:
    print(f"{a}:{b} :: {c}:?  ->", analogy(a, b, c))

In [ ]:
# Visualize with PCA (a 2-D projection of the 20-D space).
def pca_2d(M):
    Mc = M - M.mean(axis=0)
    _, _, Vt = np.linalg.svd(Mc, full_matrices=False)
    return Mc @ Vt[:2].T

show = male_royal + female_royal + male_common + female_common + animals + fruits + ["he", "she", "his", "her", "throne", "market"]
P = pca_2d(E_norm[[w2i[w] for w in show]])
colors = (["tab:blue"] * 2 + ["tab:red"] * 2 + ["tab:cyan"] * 2 + ["tab:pink"] * 2 +
          ["tab:green"] * 4 + ["tab:orange"] * 4 + ["grey"] * 6)
plt.figure(figsize=(8, 6))
plt.scatter(P[:, 0], P[:, 1], c=colors)
for (x, y), w in zip(P, show):
    plt.annotate(w, (x, y), xytext=(4, 4), textcoords="offset points")
plt.title("Word2vec trained from scratch on the toy corpus (PCA)"); plt.show()

plt.figure(figsize=(5, 3)); plt.plot(range(1, len(history) + 1), history, "o-")
plt.xlabel("epoch"); plt.ylabel("J"); plt.title("Training loss"); plt.show()

**What just happened?** The model only ever tried to predict neighboring words, yet:
* Animals cluster together, fruits cluster together, royals cluster together.
* There is a consistent **direction** for gender: *king − man + woman ≈ queen*.

This is the "astounding" result of the lecture: **meaning emerges from prediction.** Slide 7 of Lecture 2 calls it "Magic!"

#### 🔎 Beyond the slides: the same thing in PyTorch
In practice you never write gradients by hand; you define the forward computation and let autograd differentiate it (Lecture 3 explains how). Here is the identical model. Note `nn.Embedding` stores one **row** per word, exactly like our `Vc` and `U`.

In [ ]:
# [PyTorch] Same naive-softmax skip-gram, with autograd.
import torch
import torch.nn as nn
import torch.nn.functional as F

class SkipGramNaiveSoftmax(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)    # v_w
        self.outside = nn.Embedding(vocab_size, dim)   # u_w
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)
        nn.init.uniform_(self.outside.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        v_c = self.center(center_ids)                  # (B, d)
        return v_c @ self.outside.weight.T             # (B, V) logits = u_w^T v_c

torch.manual_seed(0)
model = SkipGramNaiveSoftmax(V_size, 20)
opt = torch.optim.SGD(model.parameters(), lr=0.5)
data = torch.tensor(pairs, dtype=torch.long)

for epoch in range(5):
    perm = torch.randperm(len(data))
    total = 0.0
    for k in range(0, len(data), 128):
        batch = data[perm[k:k + 128]]
        logits = model(batch[:, 0])
        loss = F.cross_entropy(logits, batch[:, 1])  # = mean of -log softmax(logits)[o]
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(batch)
    print(f"epoch {epoch+1}: J = {total / len(data):.4f}")

---
## 8. Looking at real word vectors

The lecture ends by exploring pretrained **GloVe** vectors in a Jupyter notebook (GloVe is covered in Lecture 2; for exploring, it behaves like word2vec). The cell below reproduces that exploration with `gensim`. It downloads ~130MB the first time.

Things to try: `most_similar("banana")`, analogies like `king − man + woman`, `japan:japanese :: australia:?`, `tall:tallest :: long:?`, and look at which analogies **fail**.

In [ ]:
# [Optional] needs: pip install gensim   (and internet for the download)
try:
    import gensim.downloader as api
    glove = api.load("glove-wiki-gigaword-100")      # 400k words, 100-d

    print("most similar to 'banana':", [w for w, _ in glove.most_similar("banana", topn=6)])
    print("most similar to 'croissant':", [w for w, _ in glove.most_similar("croissant", topn=6)])

    def glove_analogy(a, b, c):
        # a : b :: c : ?
        return glove.most_similar(positive=[b, c], negative=[a], topn=1)[0][0]

    for a, b, c in [("man", "king", "woman"), ("japan", "japanese", "australia"),
                    ("australia", "beer", "france"), ("tall", "tallest", "long"),
                    ("good", "fantastic", "bad"), ("obama", "clinton", "reagan")]:
        print(f"{a}:{b} :: {c}:{glove_analogy(a, b, c)}")

    words = ["coffee", "tea", "beer", "wine", "brandy", "rum", "champagne", "water",
             "spain", "france", "germany", "italy", "japan", "china",
             "school", "college", "university", "institute"]
    P = pca_2d(np.array([glove[w] for w in words]))
    plt.figure(figsize=(8, 6)); plt.scatter(P[:, 0], P[:, 1])
    for (x, y), w in zip(P, words):
        plt.annotate(w, (x, y), xytext=(4, 4), textcoords="offset points")
    plt.title("GloVe 100-d vectors (PCA)"); plt.show()
except Exception as e:
    print("Skipping pretrained-vector demo:", e)

---
## 9. 🔎 Beyond the slides: what an ML engineer should also know

### 9.1 An embedding lookup *is* a matrix multiplication with a one-hot vector
The lecture connects one-hot vectors and dense vectors through this identity. If $E\in\mathbb{R}^{|V|\times d}$ holds one row per word, then $e_i^\top E = E[i]$. Frameworks never materialize the one-hot vector; they just index the row (`nn.Embedding`). Conceptually though, **an embedding layer is a linear layer applied to one-hot inputs**, which is why its weights are learned by backprop like any other layer. Lecture 2 writes this as $x = Le$.

In [ ]:
E_small = rng.normal(size=(5, 3))       # 5 words, 3 dims
i = 2
e_i = np.zeros(5); e_i[i] = 1.0
print("one-hot @ E :", e_i @ E_small)
print("E[i]        :", E_small[i])
print("identical   :", np.allclose(e_i @ E_small, E_small[i]))

### 9.2 Limitations of word2vec-style (static) embeddings, and what came next
| Limitation | What fixes it | Where |
|---|---|---|
| **One vector per word type**, so "bank" (river) = "bank" (money) | Contextual embeddings (ELMo, BERT): a different vector per *occurrence* | L2 §senses, L9 |
| **Unknown / rare words** get no or bad vectors | Subword models: fastText (character n-grams), BPE/WordPiece tokenization | L9 |
| Window is small, **word order is ignored** (bag of words) | Sequence models: RNNs, then Transformers | L5–L8 |
| Embeddings **absorb social biases** in the data (e.g., gender stereotypes in occupation words) | Bias measurement and mitigation (Bolukbasi et al. 2016; Caliskan et al. 2017) | L17 |

### 9.3 Practical details of the real word2vec (Mikolov et al. 2013b)
* **Negative sampling** replaces the expensive softmax (Lecture 2).
* **Subsampling of frequent words:** a word with frequency $f(w)$ is dropped with probability $1-\sqrt{t/f(w)}$ (with $t\approx10^{-5}$). "the" carries little information about its neighbors, and dropping it speeds training *and* improves vectors.
* **Dynamic window:** the effective window size is sampled uniformly in $[1,m]$, so nearer words are weighted more.
* **Learning rate** decays linearly to ~0 over training.
* Typical hyperparameters: $d=300$, window 5–10, 5–20 negative samples, a few epochs over billions of tokens.

### 9.4 Do embeddings still matter in the LLM era?
Yes, in two ways:
1. **Every Transformer starts with an embedding table** (one row per token) learned exactly by backprop through a softmax prediction objective. GPT's training objective (predict the next token with a softmax over the vocabulary) is a direct descendant of the objective you just implemented.
2. **"Embeddings" for retrieval/search** (RAG, semantic search) are dense vectors compared with cosine similarity, the same geometry as this lecture, just produced by a Transformer for whole sentences.

---
## 10. Summary

| Concept | Key formula / fact |
|---|---|
| Distributional hypothesis | A word is characterized by its contexts (Firth 1957) |
| One-hot problem | All pairs orthogonal → no similarity; dimension = $\lvert V\rvert$ |
| Skip-gram likelihood | $L(\theta)=\prod_t\prod_{j\neq0}P(w_{t+j}\mid w_t;\theta)$ |
| Objective | $J(\theta)=-\frac1T\sum_t\sum_{j\neq0}\log P(w_{t+j}\mid w_t)$ |
| Probability model | $P(o\mid c)=\dfrac{\exp(u_o^\top v_c)}{\sum_w\exp(u_w^\top v_c)}$ |
| Gradient wrt center | $\nabla_{v_c}J = -u_o + \sum_x P(x\mid c)u_x = U^\top(\hat y - y)$ |
| Gradient wrt outside | $\nabla_{U}J = (\hat y - y)v_c^\top$ |
| Gradient descent | $\theta \leftarrow \theta - \alpha\nabla_\theta J$; use mini-batch SGD in practice |

### Exercises
1. **Window size.** Retrain the toy model with `m=1` and `m=4` (edit the `skipgram_pairs` call). How do nearest neighbors change? (Small windows tend to capture more *syntactic*/functional similarity; larger windows capture more *topical* similarity.)
2. **Vector dimension.** Try `d=2` and `d=100`. Does the analogy still work at `d=2`?
3. **Only center vectors.** Use `Vc` instead of `(Vc+U)/2` for similarity. Any difference?
4. **Derivation.** Derive $\partial J/\partial u_w$ yourself without looking, then check against §5.2.
5. **Hard mode.** Implement the **CBOW** variant (predict the center word from the *average* of context vectors) and compare. (Lecture 2 introduces CBOW.)

### References
* Mikolov et al. 2013a, *Efficient Estimation of Word Representations in Vector Space*; 2013b, *Distributed Representations of Words and Phrases and their Compositionality*.
* CS224N lecture notes 1 (word vectors) on the course website.
* Rong 2014, *word2vec Parameter Learning Explained* (excellent step-by-step derivations).
* Jurafsky & Martin, *Speech and Language Processing* (3rd ed. draft), Ch. 6 "Vector Semantics and Embeddings".

**Next:** Lecture 2, negative sampling, count-based methods (SVD, GloVe), evaluating vectors, word senses, and the jump to neural classifiers.